# Nara — Server Video Avatar Ditto TalkingHead + Edge TTS (`id-ID-GadisNeural`)

Notebook ini menjalankan **Ditto TalkingHead** (Apache-2.0) + **Edge TTS Bahasa Indonesia** dan mengeksposnya lewat HTTPS (`https://xxxx.trycloudflare.com`) yang langsung kompatibel dengan `novandafadly/narrator-mc`.

**Langkah cepat:**
1. Pastikan Runtime GPU aktif (`Runtime` → `Change runtime type` → **T4** / **L4** / **A100**).
2. Jalankan Sel 1–2 untuk instalasi & unduh model.
3. Jalankan Sel 3 untuk mengunggah 1 foto potret Nara (`nara.jpg` / `nara.png`).
4. Jalankan Sel 4 untuk menghasilkan & mengunduh `nara_idle.mp4` + `nara_poster.jpg` (simpan ke `public/avatar/` di repo `narrator-mc`).
5. Jalankan Sel 5 lalu salin URL `https://xxxx.trycloudflare.com` ke **Panel Operator** kanan atas di Nara → klik **Simpan & Cek**!

In [ ]:
#@title 1. Cek GPU & Clone Repo (Ditto TalkingHead + Nara Server)
!nvidia-smi

import os, pathlib

if not os.path.exists('/content/ditto-talkinghead'):
    !git clone --depth 1 https://github.com/antgroup/ditto-talkinghead.git /content/ditto-talkinghead

if not os.path.exists('/content/narrator-mc'):
    !git clone --depth 1 https://github.com/novandafadly/narrator-mc.git /content/narrator-mc
else:
    !git -C /content/narrator-mc pull --ff-only

!pip install -q -r /content/narrator-mc/gpu-server/requirements.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('Setup selesai!')

In [ ]:
#@title 2. Unduh Checkpoint Ditto dari HuggingFace
#@markdown Pilih `pytorch` untuk kompatibilitas universal (T4/L4/A100), atau `trt_ampere` jika memakai Colab Pro L4/A100 dengan TensorRT.
BACKEND_MODE = "pytorch" #@param ["pytorch", "trt_ampere"]

from huggingface_hub import snapshot_download

patterns = ["ditto_cfg/*", "ditto_pytorch/**"]
if BACKEND_MODE == "trt_ampere":
    patterns.append("ditto_trt_Ampere_Plus/**")

ckpt_dir = "/content/ditto-talkinghead/checkpoints"
snapshot_download(
    repo_id="digital-avatar/ditto-talkinghead",
    local_dir=ckpt_dir,
    allow_patterns=patterns,
)
print("Checkpoint siap di:", ckpt_dir)

In [ ]:
#@title 3. Unggah Foto Potret Nara (1 foto hadap depan, resolusi jelas)
from google.colab import files
import shutil, pathlib

uploaded = files.upload()
assert uploaded, "Belum ada file yang diunggah!"
first_name = next(iter(uploaded.keys()))
ext = pathlib.Path(first_name).suffix.lower() or ".jpg"
SOURCE_IMAGE = f"/content/nara_source{ext}"
shutil.move(first_name, SOURCE_IMAGE)
print("Foto sumber Nara tersimpan di:", SOURCE_IMAGE)

In [ ]:
#@title 4. Buat & Unduh Aset Idle Seamless (nara_idle.mp4 & nara_poster.jpg)
#@markdown Letakkan kedua file hasil unduhan ini ke folder `public/avatar/` di repo `narrator-mc`.
SAMPLING_TIMESTEPS = 15 #@param {type:"slider", min:10, max:30, step:5}
MAX_SIZE = 1024 #@param [768, 1024]

!python3 /content/narrator-mc/gpu-server/ditto_server.py \
  --ditto-dir /content/ditto-talkinghead \
  --source-image "{SOURCE_IMAGE}" \
  --backend auto \
  --sampling-timesteps {SAMPLING_TIMESTEPS} \
  --max-size {MAX_SIZE} \
  --assets-dir /content/nara_assets \
  --generate-idle-only

from google.colab import files
files.download("/content/nara_assets/nara_idle.mp4")
files.download("/content/nara_assets/nara_poster.jpg")

In [ ]:
#@title 5. Jalankan Server Ditto + Cloudflare Tunnel HTTPS (Biarkan sel ini terus berjalan)
import re, subprocess, time

cf_log = open("/tmp/cloudflared.log", "w")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=cf_log,
    stderr=subprocess.STDOUT,
)

tunnel_url = None
for _ in range(25):
    time.sleep(1)
    text = open("/tmp/cloudflared.log").read()
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
    if m:
        tunnel_url = m.group(0)
        break

print("=" * 64)
print("  URL HTTPS SERVER NARA (DITTO):", tunnel_url or "Lihat /tmp/cloudflared.log")
print("  Tempel URL di atas ke Panel Operator kanan atas narrator-mc!")
print("=" * 64)

!python3 /content/narrator-mc/gpu-server/ditto_server.py \
  --ditto-dir /content/ditto-talkinghead \
  --source-image "{SOURCE_IMAGE}" \
  --backend auto \
  --sampling-timesteps {SAMPLING_TIMESTEPS} \
  --max-size {MAX_SIZE} \
  --assets-dir /content/nara_assets \
  --port 8000